<a href="https://colab.research.google.com/github/chaitrali-gundolkar/Zepto-SQL-Data-Analysis/blob/main/Churn_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [95]:
import numpy as np
import pandas as pd
import sqlite3

In [96]:
# 1. Data Importing
excel_file = 'customer_churn_data_raw.xlsx'
conn = sqlite3.connect('customer_churn.db')

excel_data = pd.ExcelFile(excel_file)

# Loop through each sheet and store as separate table
for sheet in excel_data.sheet_names:
    df = pd.read_excel(excel_file, sheet_name=sheet) # Read sheet into dataframe
    # Write dataframe to SQLite table
    df.to_sql(
        name=sheet,          # Table name = Sheet name
        con=conn,
        if_exists='replace', # Replace table if already exists
        index=False
    )

# Close connection
conn.close()

print("All sheets successfully converted into SQLite tables.")

All sheets successfully converted into SQLite tables.


In [97]:
# Data Import: db file to pandas, storing each table to a separate df

# Connect to SQLite database
conn = sqlite3.connect('customer_churn.db')

# sql query to Get all table names
sql_query = """
        SELECT name
        FROM sqlite_master
        WHERE type='table';
"""

# read sql query in pandas
tables = pd.read_sql(sql_query, conn)

# create dataframe for each table
for table_name in tables['name']:
    df = pd.read_sql(f"SELECT * FROM {table_name}", conn) # Read table into dataframe
    globals()[f"df_{table_name}"] = df                    # Create dynamic dataframe name
    print(f"Created dataframe: df_{table_name}")

# Close connection
conn.close()

Created dataframe: df_db_customer
Created dataframe: df_db_subscription
Created dataframe: df_db_support


In [98]:
# Print table names and column names
conn = sqlite3.connect('customer_churn.db')

for table_name in tables['name']:
    print(f"\nTable Name: {table_name}")
    # Get column information
    columns_query = f"PRAGMA table_info({table_name});"
    columns = pd.read_sql(columns_query, conn)
    print("Columns:")
    print(columns['name'].tolist())

# Close connection
conn.close()


Table Name: db_customer
Columns:
['customerid', 'name', 'country', 'state', 'gender', 'dob', 'interests', 'pincode']

Table Name: db_subscription
Columns:
['customerid', 'subscription_start_date', 'subscription_type', 'renewal_date', 'plan_type', 'contract_type', 'cancellation_date', 'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score']

Table Name: db_support
Columns:
['customerid', 'complaint_date', 'escalations', 'csat_score', 'col_1', 'comment']


In [99]:
#2. Data Cleaning

df_db_customer.head(2)


,customerid,name,country,state,gender,dob,interests,pincode
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12 00:00:00,travel,None
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23 00:00:00,None,None


In [100]:
df_db_customer.tail()

,customerid,name,country,state,gender,dob,interests,pincode
16,0020-JDNXP,rikim,India,Meghalaya,Female,1994-08-19 00:00:00,None,None
17,0021-IKXGC,vishakha,India,Rajasthan,Female,2000-09-02 00:00:00,None,None
18,0022-TCJCI,raghvendra,India,Telangana,Male,1983-12-30 00:00:00,None,None
19,0023-HGHWL,rishabh,India,Uttar Pradesh,Men,1991-05-14 00:00:00,None,None
20,0023-UYUPN,sudevi,India,Maharashtra,Women,1977-10-06 00:00:00,None,None


In [101]:
df_db_customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   customerid  21 non-null     object
 1   name        21 non-null     object
 2   country     18 non-null     object
 3   state       21 non-null     object
 4   gender      21 non-null     object
 5   dob         21 non-null     object
 6   interests   4 non-null      object
 7   pincode     0 non-null      object
dtypes: object(8)
memory usage: 1.4+ KB


In [102]:
df_db_customer.shape

(21, 8)

In [103]:
df_db_customer.columns

Index(['customerid', 'name', 'country', 'state', 'gender', 'dob', 'interests',
       'pincode'],
      dtype='object')

In [104]:
# 1. rename the column name :name ->customer_name
# 2. have to fill null values in country table
# 3. drop the tables -> pincode and interests
# 4. change the data type of object column
# 5. Data standerdization of gender column


In [105]:
# 1. rename the column name :name ->customer_name
df_db_customer.rename(columns={'name':'customer_name'},inplace=True)

In [106]:
# 2. have to fill null values in country table
df_db_customer['country'].isna().sum()
df_db_customer[df_db_customer['country'].isna()]


,customerid,customer_name,country,state,gender,dob,interests,pincode
5,0013-MHZWF,durga,None,Delhi,Women,1988-12-10 00:00:00,None,None
8,0015-UOCOJ,maya,None,Kathmandu,Women,1985-07-07 00:00:00,None,None
12,0018-NYROU,chitra,None,Telangana,Female,2004-12-01 00:00:00,None,None


In [107]:
# country and state - unique value pair

# Creating state → country map from non-null rows
state_country_mapping = df_db_customer.dropna(subset=['country']).set_index('state')['country'].to_dict()


In [108]:
# Fill the missing country using State
df_db_customer['country'] = df_db_customer['country'].fillna(df_db_customer['state'].map(state_country_mapping))

In [109]:
df_db_customer[df_db_customer['country'].isna()] # no null value in country col

,customerid,customer_name,country,state,gender,dob,interests,pincode


In [110]:
# 3. drop the tables -> pincode and interests
df_db_customer.drop(columns={'pincode','interests'},inplace=True)

In [111]:
# 4. change the data type of object column
df_db_customer['dob']=pd.to_datetime(df_db_customer['dob'])

In [112]:
# 5. Data standerdization of gender column
#df_db_customer['gender'].unique()
df_db_customer['gender']=df_db_customer['gender'].replace({'Male':'M', 'Female':'F', 'Women':'F', 'Men':'M'})


In [113]:
df_db_customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   customerid     21 non-null     object        
 1   customer_name  21 non-null     object        
 2   country        21 non-null     object        
 3   state          21 non-null     object        
 4   gender         21 non-null     object        
 5   dob            21 non-null     datetime64[ns]
dtypes: datetime64[ns](1), object(5)
memory usage: 1.1+ KB


In [114]:
df_db_subscription.head()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,None,None,13.99,627,12
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,91
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,None,None,6.99,210,34
3,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,None,None,22.99,1725,8
4,0013-EXCHZ,2023-01-05,Refferal,2024-01-05,Standard,Monthly,2024-02-28,Too expensive,13.99,195,88


In [115]:
df_db_subscription.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customerid               21 non-null     object 
 1   subscription_start_date  21 non-null     object 
 2   subscription_type        21 non-null     object 
 3   renewal_date             21 non-null     object 
 4   plan_type                21 non-null     object 
 5   contract_type            21 non-null     object 
 6   cancellation_date        6 non-null      object 
 7   cancellation_reason      6 non-null      object 
 8   monthly_charges          21 non-null     float64
 9   cltv                     21 non-null     int64  
 10  churn_score              21 non-null     int64  
dtypes: float64(1), int64(2), object(8)
memory usage: 1.9+ KB


In [116]:
#1. change datatype of subscription_start_date ,renewal_date and cancellation_date --date
date_col=['subscription_start_date','renewal_date','cancellation_date']
df_db_subscription[date_col]=df_db_subscription[date_col].apply(pd.to_datetime)

In [117]:
df_db_subscription.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   customerid               21 non-null     object        
 1   subscription_start_date  21 non-null     datetime64[ns]
 2   subscription_type        21 non-null     object        
 3   renewal_date             21 non-null     datetime64[ns]
 4   plan_type                21 non-null     object        
 5   contract_type            21 non-null     object        
 6   cancellation_date        6 non-null      datetime64[ns]
 7   cancellation_reason      6 non-null      object        
 8   monthly_charges          21 non-null     float64       
 9   cltv                     21 non-null     int64         
 10  churn_score              21 non-null     int64         
dtypes: datetime64[ns](3), float64(1), int64(2), object(5)
memory usage: 1.9+ KB


In [118]:
df_db_support.head()

,customerid,complaint_date,escalations,csat_score,col_1,comment
0,0003-MKNFE,2024-08-28 00:00:00,N,60,None,service issue
1,0003-MKNFE,2024-08-28 00:00:00,Y,10,None,demaned refund
2,0013-EXCHZ,2024-01-20 00:00:00,Y,20,None,None
3,0013-MHZWF,2025-03-18 00:00:00,N,90,None,guidance to renew
4,0013-SMEOE,2024-11-01 00:00:00,N,30,None,None


In [119]:
df_db_support.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   customerid      9 non-null      object
 1   complaint_date  9 non-null      object
 2   escalations     9 non-null      object
 3   csat_score      9 non-null      int64 
 4   col_1           0 non-null      object
 5   comment         4 non-null      object
dtypes: int64(1), object(5)
memory usage: 564.0+ bytes


In [120]:
#1. change data type of complaint_date object to date
#2. drop the columns->col1 ,comment


In [121]:
#1. change data type of complaint_date object to date
df_db_support['complaint_date']=pd.to_datetime(df_db_support['complaint_date'])

In [122]:
#2. drop the columns->col1 ,comment
df_db_support.drop(columns={'col_1','comment'},inplace=True)

In [123]:
df_db_support.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   customerid      9 non-null      object        
 1   complaint_date  9 non-null      datetime64[ns]
 2   escalations     9 non-null      object        
 3   csat_score      9 non-null      int64         
dtypes: datetime64[ns](1), int64(1), object(2)
memory usage: 420.0+ bytes


In [124]:
#3. Feature Engineering & Data Analysis

In [125]:
df_db_subscription.head(1)

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaT,None,13.99,627,12


In [126]:
#Customer is churned if cancellation_date is not null

df_db_subscription['churn_flag']=np.where(df_db_subscription['cancellation_date'].notna(), 1, 0)

In [127]:
df_db_subscription.head()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score,churn_flag
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaT,None,13.99,627,12,0
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,91,1
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,NaT,None,6.99,210,34,0
3,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,NaT,None,22.99,1725,8,0
4,0013-EXCHZ,2023-01-05,Refferal,2024-01-05,Standard,Monthly,2024-02-28,Too expensive,13.99,195,88,1


In [128]:
#Merge DataFrames--Joins
df_db_customer.shape

(21, 6)

In [129]:
#First fix support table then merge
df=(df_db_subscription.merge(df_db_customer,on='customerid',how='left').merge(df_db_support,on='customerid',how='left'))

In [130]:
df_db_subscription.shape

(21, 12)

In [131]:
df.shape

(23, 20)

In [132]:
df_db_customer['customerid'].nunique()

21

In [133]:
df_db_subscription['customerid'].nunique()

21

In [134]:
df_db_support['customerid'].nunique()

7

In [135]:
df_db_support['customerid'].size

9

In [136]:
df_db_support['complaint_count']=df_db_support.groupby('customerid')['customerid'].transform('count')

In [137]:
df_db_support

,customerid,complaint_date,escalations,csat_score,complaint_count
0,0003-MKNFE,2024-08-28,N,60,2
1,0003-MKNFE,2024-08-28,Y,10,2
2,0013-EXCHZ,2024-01-20,Y,20,1
3,0013-MHZWF,2025-03-18,N,90,1
4,0013-SMEOE,2024-11-01,N,30,1
5,0017-IUDMW,2024-04-10,Y,25,1
6,0019-EFAEP,2024-09-27,Y,30,1
7,0022-TCJCI,2024-09-13,Y,10,2
8,0022-TCJCI,2024-09-14,N,90,2


In [138]:
df_db_support=df_db_support.sort_values('complaint_date').drop_duplicates('customerid',keep='last')

In [139]:
df_db_support['customerid'].size

7

In [140]:
#merge
df=(df_db_subscription.merge(df_db_customer,on='customerid',how='left').merge(df_db_support,on='customerid',how='left'))

In [141]:
df.shape

(21, 21)

In [142]:
df.to_csv('Exported_churn_data.csv',index=False)

In [143]:
#3. Data Analysis


In [144]:
# 1. Churn Rate

In [145]:
df.columns

Index(['customerid', 'subscription_start_date', 'subscription_type',
       'renewal_date', 'plan_type', 'contract_type', 'cancellation_date',
       'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score',
       'churn_flag', 'customer_name', 'country', 'state', 'gender', 'dob',
       'complaint_date', 'escalations', 'csat_score', 'complaint_count'],
      dtype='object')

In [146]:
churn_rate=df['churn_flag'].mean()*100
print('Churn_Rate',churn_rate.round(2))

Churn_Rate 28.57


In [147]:
# 2. Retenion Rate

In [148]:
retenion_rate=100-churn_rate
print('Retenion_Rate',round(retenion_rate,2))

Retenion_Rate 71.43


In [149]:
# 3. Churn by Plan type

In [150]:
Churn_plan_type=df.groupby('plan_type')['churn_flag'].mean().mul(100).round(2).reset_index()
print(Churn_plan_type)

  plan_type  churn_flag
0     Basic       60.00
1   Premium       14.29
2  Standard       22.22


In [151]:
# 4 a. Churn by state + sum(revenue) & count of users

In [152]:
result = df.groupby('state').agg({
    'churn_flag': 'sum',
    'monthly_charges': 'sum',
    'customerid': 'count'
})

print(result)


               churn_flag  monthly_charges  customerid
state                                                 
Delhi                   1            52.96           4
Karnataka               2            20.98           2
Kathmandu               0            20.98           2
Maharashtra             0            50.97           3
Meghalaya               2            42.97           3
Nagaland                0            22.99           1
Rajasthan               0            36.98           2
Telangana               1            30.98           2
Uttar Pradesh           0           115.98           2


In [153]:
# 4 b. Churn by subscription type + sum(revenue) & count of users

In [154]:
df.groupby('subscription_type').agg({'churn_flag': 'sum',
    'monthly_charges': 'sum',
    'customerid': 'count'
})

,churn_flag,monthly_charges,customerid
subscription_type,,,
Organic,0,145.91,9
Paid,1,174.94,6
Refferal,5,74.94,6


In [155]:
# 5. ARPU - Avg Revenue per user
ARPU=df['monthly_charges'].mean()
print(ARPU.round(2))

18.85


In [156]:
# 6. calculate customer age


In [157]:
from datetime import datetime

today = datetime.now()

df['age'] = df['dob'].map(
    lambda x: today.year - pd.to_datetime(x).year
)

print(df[['customer_name', 'dob', 'age']])

   customer_name        dob  age
0         keshav 1982-04-12   44
1         raghav 1995-11-23   31
2         lalita 1978-02-15   48
3          mohan 2001-08-30   25
4           mira 1990-05-05   36
5          durga 1988-12-10   38
6           mina 1976-09-21   50
7          madan 1999-03-14   27
8           maya 1985-07-07   41
9          arjun 1993-10-29   33
10         shiva 1997-01-22   29
11     rangadevi 1981-06-18   45
12        chitra 2004-12-01   22
13          raju 1992-04-25   34
14        Madhav 1979-11-11   47
15       parvati 1986-02-28   40
16         rikim 1994-08-19   32
17      vishakha 2000-09-02   26
18    raghvendra 1983-12-30   43
19       rishabh 1991-05-14   35
20        sudevi 1977-10-06   49


In [158]:
# 6. Avg Customer Tenure
# count of days users has used our service : cancellation date else current date

In [161]:
today=pd.Timestamp.today()

df['tenure_days'] = np.where(
        df['cancellation_date'].notna(),

    (df['cancellation_date'] - df['subscription_start_date']).dt.days,

    (today - df['subscription_start_date']).dt.days
)

avg_tenure = df['tenure_days'].mean()
print("Avg Tenure (Days) = ", round(avg_tenure),0)

Avg Tenure (Days) =  1582 0


In [160]:
df.columns

Index(['customerid', 'subscription_start_date', 'subscription_type',
       'renewal_date', 'plan_type', 'contract_type', 'cancellation_date',
       'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score',
       'churn_flag', 'customer_name', 'country', 'state', 'gender', 'dob',
       'complaint_date', 'escalations', 'csat_score', 'complaint_count',
       'age'],
      dtype='object')

In [ ]:
# 7. Revenue at risk - revenue lost from churned users

In [162]:
revenue_at_risk = df.loc[df['churn_flag']==1, 'monthly_charges'].sum()
print("Revenue at Risk (Rs 'K') =", revenue_at_risk)

Revenue at Risk (Rs 'K') = 73.94


In [ ]:
# 8. Esclation Rate

In [164]:
escalation_rate = (df['escalations']=='Y').mean()*100
print("Esclation Rate = ", round(escalation_rate, 2), "%")

Esclation Rate =  19.05 %


In [165]:
# 9. Avg Complaint Per User
avg_complaints = df['complaint_count'].sum() / df['customerid'].nunique()
print("Avg Compliants Per User = ", round(avg_complaints, 2))

Avg Compliants Per User =  0.43


In [166]:
# 10. Correlation Esclation vs Churn
df['escalations'] = np.where(df['escalations'] == 'Y', 1, 0) # encoding string to int type
corr_df = df[['escalations', 'churn_flag']].dropna()

correlation = corr_df['escalations'].corr(df['churn_flag'])
print("Correlation between esclation vs churn is = ", round(correlation,2))

Correlation between esclation vs churn is =  0.77


In [167]:
# 11. Create a column using existing col - Churn risk
conditions = [
        (df['churn_score'] < 50),
        (df['churn_score'] >= 50) & (df['churn_score'] < 70),
        (df['churn_score'] >= 70)
]

choices = ['low', 'med', 'high']

df['churn_risk'] = np.select(conditions, choices, default='unkown')

In [ ]:
#Visualization using matplotlib and seaborn


In [168]:
#Create copy of df
df_visual=df.copy()

In [169]:
df.columns

Index(['customerid', 'subscription_start_date', 'subscription_type',
       'renewal_date', 'plan_type', 'contract_type', 'cancellation_date',
       'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score',
       'churn_flag', 'customer_name', 'country', 'state', 'gender', 'dob',
       'complaint_date', 'escalations', 'csat_score', 'complaint_count', 'age',
       'tenure_days', 'churn_risk'],
      dtype='object')